# NetOps-7B: Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwccie/netops-7b/blob/main/notebooks/03_evaluation.ipynb)

Evaluate NetOps-7B against the base Qwen 2.5 7B model.

## Evaluation Dimensions

1. **Config Syntax Validity** — Regex-based structural parsing
2. **Troubleshooting Accuracy** — 500-question test set
3. **Explanation Quality** — ROUGE/BERTScore + domain metrics
4. **Base Model Comparison** — Side-by-side scoring

In [ ]:
!pip install -q git+https://github.com/cwccie/netops-7b.git
!pip install -q unsloth rouge-score bert-score

In [ ]:
from netops_7b.eval.harness import EvaluationHarness

# Generate test set
test_path = EvaluationHarness.generate_test_set("data/test.jsonl", size=500)
print(f"Test set saved to: {test_path}")

In [ ]:
# Evaluate fine-tuned model
harness = EvaluationHarness(model_path="outputs/netops-7b-qlora/final")
harness.load_model()
summary = harness.run("data/test.jsonl")

print(f"\nOverall Scores:")
for metric, score in sorted(summary.overall_scores.items()):
    print(f"  {metric}: {score:.4f}")

In [ ]:
# Run benchmark comparison
from netops_7b.eval.benchmark import BenchmarkRunner

runner = BenchmarkRunner(
    base_model="Qwen/Qwen2.5-7B-Instruct",
    finetuned_model="outputs/netops-7b-qlora/final",
)
result = runner.run("data/test.jsonl")

print(result.to_markdown_table())

In [ ]:
# Save results
harness.save_results("outputs/eval_results.json")
runner.save(result, "outputs/benchmark.json")
print("Results saved.")